# W04-面向对象

In [7]:
from math import pi

class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return pi * self.radius * self.radius
        

circle1 = Circle(10)
circle1.area()

## 实例 / 类 / 静态

In [16]:
class Student:
    school_name = "Computer Science School" # 类变量：所有学生都共享

    def __init__(self, name, score):
        self.name = name  # 实例数据：每个学生不同
        self.score = score

    def show_me(self):
        # need self
        print(f"My name is {self.name}")

    @staticmethod
    def score_to_grade(score):
        # just use score, don't need self or cls
        if score >= 90:
            return "A"
        elif score >= 80:
            return "B"
        elif score >= 70:
            return "C"
        elif score >= 60:
            return "D"
        else:
            return "F"

    @classmethod
    def about_school(cls):
        # need cls
        print(f"Welocome to {cls.school_name}")
            

In [ ]:
s1 = Student("小明", 95)
s1.show_me()
Student.about_school()
Student.score_to_grade(95)

## 公有 / 私有：下划线约定

In [20]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner          # 公有
        self._secret = "内部用"      # 单下划线：约定"别动我"
        self.__pin = "1234"         # 双下划线：名称改写

a = Account("李雷", 100)
print(a.owner)              # 李雷
print(a._secret)            # 仍然能访问，只是约定
# print(a.__pin)            # ❌ AttributeError
print(a._Account__pin)      # 1234（双下划线被改成 _类名__name）


李雷
内部用
1234


## 多态：同一个调用，不同表现 

In [21]:
class Dog:
    def speak(self):
        return "汪汪"

class Cat:
    def speak(self):
        return "喵喵"

def animal_talk(animal):
    # 只要对象有 speak 方法，就能调用
    return animal.speak()

print(animal_talk(Dog()))  # 汪汪
print(animal_talk(Cat()))  # 喵喵


汪汪
喵喵


## 实战 1：完整 Vector 类

In [26]:
import math
from dataclasses import dataclass

@dataclass
class Vec:
    x: float
    y: float

    def __add__(self, other):
        return Vec(self.x + other.x, self.y + other.y)
    def __sub__(self, other):
        return Vec(self.x - other.x, self.y - other.y)
    def __mul__(self, k):
        return Vec(self.x * k, self.y * k)
    def length(self):
        return math.hypot(self.x, self.y)

a = Vec(1, 2); b = Vec(4, 6)
print(a + b)              # Vec(x=5, y=8)
print((b - a).length())   # 5.0
print(a * 3)              # Vec(x=3, y=6)


Vec(x=5, y=8)
5.0
Vec(x=3, y=6)


## 实战 2：银行账户体系

In [33]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance

    def deposit(self, n):
        self._balance += n

    def withdraw(self, n):
        if self._balance < n:
            raise ValueError("余额不足")
        self._balance -= n

    @property
    def balance(self):
        return self._balance4

    def __repr__(self):
        return f"<{type(self).__name__} {self.owner} ${self._balance}>"


class SavingsAccount(Account):
    def __init__(self, owner, balance=0, rate=0.03):
        super().__init__(owner, balance)
        self.rate = rate

    def add_interest(self):
        self._balance *= (1 + self.rate)


class CheckingAccount(Account):
    OVERDRFT = -1000
    def withdraw(self, n):
        if n > self._balance - self.OVERDRFT:
            raise ValueError("超出透支限额")
        self._balance -= n


s = SavingsAccount("李雷", 1000)
s.add_interest()
print(s)                              # <SavingsAccount 李雷 ¥1030.0>

c = CheckingAccount("韩梅梅", 500)
c.withdraw(800)                       # 允许透支
print(c)                              # <CheckingAccount 韩梅梅 ¥-300>


<SavingsAccount 李雷 $1030.0>
<CheckingAccount 韩梅梅 $-300>


## 实战 3：扑克牌发牌器

In [36]:
import random
from dataclasses import dataclass

@dataclass(frozen=True)
class Card:
    rank: str
    suit: str
    def __repr__(self):
        return f"{self.rank}{self.suit}"

class Deck:
    RANKS = list("23456789TJQKA")
    SUITS = "♠♥♦♣"

    def __init__(self):
        self.cards = [Card(r, s) for r in self.RANKS for s in self.SUITS]
        random.shuffle(self.cards)

    def deal(self, n):
        hand, self.cards = self.cards[:n], self.cards[n:]
        return hand

    def __len__(self):
        return len(self.cards)


deck = Deck()
print(deck.deal(5))
print("剩余", len(deck))


[7♥, J♠, 5♦, T♦, 4♠]
剩余 47
